# 01 Python Data Processing and Analytics
This notebook loads the NorthStar dataset, cleans inconsistent zones, joins related CSV files, creates analytical fields, produces summary tables and generates charts for the report.

In [ ]:
# Install packages if running in a fresh Colab runtime
!pip -q install pandas numpy matplotlib

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# In Colab, upload the CSV files into /content/northstar_dataset or change this path.
DATA_DIR = Path("/content/northstar_dataset")
if not DATA_DIR.exists():
    from google.colab import files
    print("Upload all CSV files from data/raw when prompted.")
    uploaded = files.upload()
    DATA_DIR.mkdir(exist_ok=True)
    for name, data in uploaded.items():
        Path(name).rename(DATA_DIR / name)
print("Using data folder:", DATA_DIR)

In [ ]:
files = {
    "orders": "orders.csv", "deliveries": "deliveries.csv", "complaints": "complaints.csv",
    "customers": "customers.csv", "drivers": "drivers.csv", "hubs": "hubs.csv",
    "incidents": "incidents.csv", "vehicles": "vehicles.csv", "app_events": "app_events.csv"
}
data = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
for name, df in data.items():
    print(name, df.shape)
    display(df.head(3))

In [ ]:
def normalise_zone(value):
    if pd.isna(value):
        return value
    return str(value).strip().title()

for df, col in [(data["orders"], "pickup_zone"), (data["orders"], "dropoff_zone"),
                (data["drivers"], "base_zone"), (data["vehicles"], "assigned_zone"),
                (data["hubs"], "zone"), (data["app_events"], "zone_context")]:
    df[col] = df[col].apply(normalise_zone)

In [ ]:
orders = data["orders"]
deliveries = data["deliveries"]
complaints = data["complaints"]
incidents = data["incidents"]

complaint_summary = complaints.groupby("order_id").agg(
    complaint_count=("complaint_id", "count"),
    total_compensation=("compensation_amount", "sum"),
    max_resolution_days=("resolution_days", "max")
).reset_index()

incident_summary = incidents.groupby("delivery_id").agg(
    incident_count=("incident_id", "count"),
    avg_incident_resolved_hours=("resolved_hours", "mean")
).reset_index()

integrated = deliveries.merge(orders, on="order_id", how="left") \
    .merge(data["hubs"], on="hub_id", how="left") \
    .merge(data["drivers"], on="driver_id", how="left") \
    .merge(data["vehicles"], on="vehicle_id", how="left") \
    .merge(complaint_summary, on="order_id", how="left") \
    .merge(incident_summary, on="delivery_id", how="left")

integrated[["complaint_count", "total_compensation", "max_resolution_days", "incident_count"]] = integrated[["complaint_count", "total_compensation", "max_resolution_days", "incident_count"]].fillna(0)
integrated["problem_delivery"] = integrated["delivery_status"].isin(["Delayed", "Failed"]).astype(int)
integrated["failed_delivery"] = (integrated["delivery_status"] == "Failed").astype(int)
print(integrated.shape)
display(integrated.head())

In [ ]:
hub_performance = integrated.groupby(["hub_id", "hub_name", "zone"]).agg(
    total_deliveries=("delivery_id", "count"),
    problem_rate=("problem_delivery", "mean"),
    failure_rate=("failed_delivery", "mean"),
    avg_rating=("customer_rating_post_delivery", "mean"),
    avg_overrides=("manual_route_override_count", "mean"),
    proof_missing_rate=("proof_of_completion_missing", "mean"),
    complaints=("complaint_count", "sum"),
    incidents=("incident_count", "sum"),
    avg_cost=("fuel_or_charge_cost", "mean")
).reset_index().sort_values("problem_rate", ascending=False)
display(hub_performance)

In [ ]:
service_performance = integrated.groupby("service_type").agg(
    total_orders=("order_id", "count"),
    problem_rate=("problem_delivery", "mean"),
    failure_rate=("failed_delivery", "mean"),
    avg_customer_rating=("customer_rating_post_delivery", "mean"),
    avg_order_value=("order_value", "mean"),
    avg_cost=("fuel_or_charge_cost", "mean"),
    complaints=("complaint_count", "sum")
).reset_index().sort_values("problem_rate", ascending=False)
display(service_performance)

In [ ]:
complaint_types = complaints["complaint_type"].value_counts().reset_index()
complaint_types.columns = ["complaint_type", "count"]
app_performance = data["app_events"].groupby("event_type").agg(
    events=("event_id", "count"),
    success_rate=("success_flag", "mean"),
    avg_latency_ms=("api_latency_ms", "mean")
).reset_index().sort_values(["success_rate", "avg_latency_ms"])
display(complaint_types)
display(app_performance)

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(hub_performance["hub_name"], hub_performance["problem_rate"])
plt.title("Problem delivery rate by hub")
plt.ylabel("Delayed or failed rate")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(service_performance["service_type"], service_performance["problem_rate"])
plt.title("Problem delivery rate by service type")
plt.ylabel("Delayed or failed rate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(complaint_types["complaint_type"], complaint_types["count"])
plt.title("Complaint types")
plt.ylabel("Number of complaints")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
correlation_cols = ["problem_delivery", "manual_route_override_count", "proof_of_completion_missing",
                    "route_distance_km", "training_score", "driver_rating", "battery_health_pct",
                    "incident_count", "customer_rating_post_delivery", "fuel_or_charge_cost"]
correlations = integrated[correlation_cols].corr()["problem_delivery"].sort_values(ascending=False)
display(correlations)

In [ ]:
OUTPUT_DIR = Path("/content/northstar_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
hub_performance.to_csv(OUTPUT_DIR / "hub_performance.csv", index=False)
service_performance.to_csv(OUTPUT_DIR / "service_performance.csv", index=False)
integrated.to_csv(OUTPUT_DIR / "integrated_delivery_analysis.csv", index=False)
print("Saved outputs to", OUTPUT_DIR)